# Dynamic INT8 Quantization

This notebook applies post training dynamic INT8 quantization to the MNLI fine tuned BERT models using TorchAO.

Each quantized model is created directly from its corresponding FP32 checkpoint. The same procedure will be applied to seeds 42, 123 and 456.

## Setup

In [1]:
import torch
import torchao

from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from torchao.quantization import(
    quantize_,
    Int8DynamicActivationInt8WeightConfig
)

print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)

W0925 22:08:26.961000 31022 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0925 22:08:26.984000 31022 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


PyTorch: 2.14.0
TorchAO: 0.18.0


## Configurations

In [2]:
SEED = 456
MAX_LENGTH = 128
BATCH_SIZE = 32

MODEL_PATH = f"../models/fp32/seed{SEED}/final"

print("Seed:", SEED)
print("Model path:", MODEL_PATH)

Seed: 456
Model path: ../models/fp32/seed456/final


## Load FP32 Model

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model_fp32 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

model_fp32.eval()

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

print("Model Loaded")
print("Labels:", model_fp32.config.id2label)

W0925 22:08:29.281000 31022 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model Loaded
Labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


## Apply Dynamic INT8 Quantization

In [4]:
import copy

model_int8 = copy.deepcopy(model_fp32)

quantize_(
    model_int8,
    Int8DynamicActivationInt8WeightConfig()
)

model_int8.eval()
print("Dynamic INT8 quantization completed")

Dynamic INT8 quantization completed


## Verify Quantization

Check how many linear layers have INT8 weights after quantization and identify any linear layers that remain in FP32.

In [5]:
int8_layers = []
fp32_layers = []

for name, module in model_int8.named_modules():
    if isinstance(module, torch.nn.Linear):
        if type(module.weight).__name__ == "Int8Tensor":
            int8_layers.append(name)
        else:
            fp32_layers.append(name)

print("INT8 linear layers:", len(int8_layers))
print("FP32 linear layers:", len(fp32_layers))

print("\nFP32 linear layers:")
for name in fp32_layers:
    print(name)

INT8 linear layers: 74
FP32 linear layers: 0

FP32 linear layers:


## Load HANS

In [6]:
from datasets import load_dataset

HANS_URL = (
    "https://raw.githubusercontent.com/tommccoy1/hans/"
    "master/heuristics_evaluation_set.txt"
)

hans = load_dataset(
    "csv",
    data_files={"validation": HANS_URL},
    delimiter="\t"
)

hans = hans["validation"]
print(hans)

Dataset({
    features: ['gold_label', 'sentence1_binary_parse', 'sentence2_binary_parse', 'sentence1_parse', 'sentence2_parse', 'sentence1', 'sentence2', 'pairID', 'heuristic', 'subcase', 'template'],
    num_rows: 30000
})


## Prepare HANS Evaluation Data
Keep the fields needed for evaluation and rename the premise and hypothesis columns to match the BERT input format

In [7]:
hans_eval = hans.rename_columns({
    "sentence1": "premise",
    "sentence2": "hypothesis"
})

print("Number of examples:", len(hans_eval))
print(hans_eval[0]["premise"])
print(hans_eval[0]["hypothesis"])
print(hans_eval[0]["gold_label"])

Number of examples: 30000
The president advised the doctor .
The doctor advised the president .
non-entailment


## Tokenize HANS


In [8]:
def tokenize_hans(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        max_length=MAX_LENGTH
    )

hans_tokenized = hans_eval.map(
    tokenize_hans,
    batched=True
)

print(hans_tokenized)

Dataset({
    features: ['gold_label', 'sentence1_binary_parse', 'sentence2_binary_parse', 'sentence1_parse', 'sentence2_parse', 'premise', 'hypothesis', 'pairID', 'heuristic', 'subcase', 'template', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 30000
})


## Create HANS Data Loader
Creating batches using dynamic padding 

In [9]:
model_columns = ["input_ids", "token_type_ids", "attention_mask"]

hans_model_inputs = hans_tokenized.select_columns(model_columns)

hans_loader = DataLoader(
    hans_model_inputs,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

print("Number of examples:", len(hans_model_inputs))
print("Number of batches:", len(hans_loader))

Number of examples: 30000
Number of batches: 938


## Run INT8 Inference on HANS

In [10]:
all_logits = []

model_int8.eval()

with torch.no_grad():
    for batch in hans_loader:
        outputs = model_int8(**batch)
        all_logits.append(outputs.logits.cpu())

int8_logits = torch.cat(all_logits, dim=0)
int8_probs = torch.softmax(int8_logits, dim=-1)
int8_predictions = int8_probs.argmax(dim=-1)

print("Logits shape:", int8_logits.shape)
print("Predictions:", len(int8_predictions))

Logits shape: torch.Size([30000, 3])
Predictions: 30000


## Convert MNLI Predictions to HANS Labels
Map 3 MNLI classes to 2 HANS classes

In [11]:
mnli_labels = {
    0: "entailment",
    1: "neutral",
    2: "contradiction"
}

hans_predictions = []

for prediction in int8_predictions.tolist():
    if prediction == 0:
        hans_predictions.append("entailment")
    else:
        hans_predictions.append("non-entailment")

print("Number of HANS predictions:", len(hans_predictions))
print("First 5 predictions:", hans_predictions[:5])

Number of HANS predictions: 30000
First 5 predictions: ['entailment', 'entailment', 'entailment', 'entailment', 'entailment']


## Create Per-Example Results

In [12]:
import pandas as pd

results_df = pd.DataFrame({
    "gold_label": hans_eval["gold_label"],
    "premise": hans_eval["premise"],
    "hypothesis": hans_eval["hypothesis"],
    "pairID": hans_eval["pairID"],
    "heuristic": hans_eval["heuristic"],
    "subcase": hans_eval["subcase"],
    "template": hans_eval["template"]
})

results_df["mnli_prediction_id"] = int8_predictions.tolist()
results_df["mnli_prediction"] = [
    mnli_labels[prediction] for prediction in int8_predictions.tolist()
]

results_df["hans_prediction"] = hans_predictions

results_df["correct"] = (
    results_df["hans_prediction"] == results_df["gold_label"]
)

results_df["prob_entailment"] = int8_probs[:, 0].tolist()
results_df["prob_neutral"] = int8_probs[:, 1].tolist()
results_df["prob_contradiction"] = int8_probs[:, 2].tolist()

results_df["prob_non_entailment"] = (
    results_df["prob_neutral"] + results_df["prob_contradiction"]
)

print(results_df.shape)
results_df.head()

(30000, 15)


,gold_label,premise,hypothesis,pairID,heuristic,subcase,template,mnli_prediction_id,mnli_prediction,hans_prediction,correct,prob_entailment,prob_neutral,prob_contradiction,prob_non_entailment
0,non-entailment,The president advised the doctor .,The doctor advised the president .,ex0,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False,0.979204,0.003886,0.016910,0.020796
1,non-entailment,The student saw the managers .,The managers saw the student .,ex1,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False,0.982234,0.007689,0.010077,0.017766
2,non-entailment,The presidents encouraged the banker .,The banker encouraged the presidents .,ex2,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False,0.980811,0.008460,0.010729,0.019189
3,non-entailment,The senators supported the actor .,The actor supported the senators .,ex3,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False,0.906312,0.027052,0.066636,0.093688
4,non-entailment,The actors avoided the bankers .,The bankers avoided the actors .,ex4,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False,0.920861,0.022574,0.056565,0.079139


## Overall HANS Results

In [13]:
overall_accuracy = results_df["correct"].mean()

accuracy_by_label = (
    results_df.groupby("gold_label")["correct"]
    .mean()
)

print(f"Overall accuracy: {overall_accuracy:.4f}")
print("\nAccuracy by label:")
print(accuracy_by_label)

Overall accuracy: 0.5328

Accuracy by label:
gold_label
entailment        0.991533
non-entailment    0.074000
Name: correct, dtype: float64


## Results by Heuristic

In [14]:
accuracy_by_heuristic = (
    results_df.groupby("heuristic")["correct"]
    .mean()
)

print("Accuracy by heuristic:")
print(accuracy_by_heuristic)

Accuracy by heuristic:
heuristic
constituent        0.5215
lexical_overlap    0.5677
subsequence        0.5091
Name: correct, dtype: float64


## Results by Heuristic and Label

In [15]:
accuracy_by_heuristic_label = (
    results_df.groupby(["heuristic", "gold_label"])["correct"]
    .mean()
    .unstack()
)

print("Accuracy by heuristic and label:")
print(accuracy_by_heuristic_label)

Accuracy by heuristic and label:
gold_label       entailment  non-entailment
heuristic                                  
constituent          0.9996          0.0434
lexical_overlap      0.9774          0.1580
subsequence          0.9976          0.0206


## Results by Subcase

In [16]:
accuracy_by_subcase = (
    results_df.groupby(["heuristic", "subcase", "gold_label"])["correct"]
    .mean()
)

print("Accuracy by subcase:")
print(accuracy_by_subcase.to_string())

Accuracy by subcase:
heuristic        subcase                         gold_label    
constituent      ce_adverb                       entailment        1.000
                 ce_after_since_clause           entailment        1.000
                 ce_conjunction                  entailment        1.000
                 ce_embedded_under_since         entailment        0.999
                 ce_embedded_under_verb          entailment        0.999
                 cn_adverb                       non-entailment    0.014
                 cn_after_if_clause              non-entailment    0.000
                 cn_disjunction                  non-entailment    0.000
                 cn_embedded_under_if            non-entailment    0.158
                 cn_embedded_under_verb          non-entailment    0.045
lexical_overlap  le_around_prepositional_phrase  entailment        0.999
                 le_around_relative_clause       entailment        0.992
                 le_conjunction        

## Save INT8 HANS Predictions

In [17]:
import os

RESULTS_DIR = "../results/hans"
os.makedirs(RESULTS_DIR, exist_ok=True)

predictions_path = f"{RESULTS_DIR}/int8_seed{SEED}_predictions.csv"

results_df.to_csv(predictions_path, index=False)

print("Saved:", predictions_path)

Saved: ../results/hans/int8_seed456_predictions.csv


## Save INT8 HANS Summary

In [18]:
import json

summary = {
    "model": "google-bert/bert-base-uncased",
    "seed": SEED,
    "precision": "dynamic_int8",
    "dataset": "HANS",
    "num_examples": len(results_df),
    "overall_accuracy": overall_accuracy,
    "by_label": accuracy_by_label.to_dict(),
    "by_heuristic": accuracy_by_heuristic.to_dict(),
    "by_heuristic_and_label": accuracy_by_heuristic_label.to_dict(),
}

summary_path = f"{RESULTS_DIR}/int8_seed{SEED}_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:", summary_path)

Saved: ../results/hans/int8_seed456_summary.json


## Load MNLI Validation Sets
Load the MNLI matched and mismatched validation sets for INT8 evaluation

In [19]:
from datasets import load_dataset

mnli = load_dataset("nyu-mll/glue", "mnli")

mnli_matched = mnli["validation_matched"]
mnli_mismatched = mnli["validation_mismatched"]

print("Matched examples:", len(mnli_matched))
print("Mismatched examples:", len(mnli_mismatched))

Matched examples: 9815
Mismatched examples: 9832


## Tokenize MNLI
Use same tokenizer

In [20]:
def tokenize_mnli(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        max_length=MAX_LENGTH
    )

mnli_matched_tokenized = mnli_matched.map(
    tokenize_mnli,
    batched=True
)

mnli_mismatched_tokenized = mnli_mismatched.map(
    tokenize_mnli,
    batched=True
)

print("Matched:", len(mnli_matched_tokenized))
print("Mismatched:", len(mnli_mismatched_tokenized))

Matched: 9815
Mismatched: 9832


## Create MNLI Dataloader
Create dynamically padded batches

In [21]:
mnli_columns = [
    "input_ids",
    "token_type_ids",
    "attention_mask",
    "label"
]

matched_inputs = mnli_matched_tokenized.select_columns(mnli_columns)
mismatched_inputs = mnli_mismatched_tokenized.select_columns(mnli_columns)

matched_loader = DataLoader(
    matched_inputs,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

mismatched_loader = DataLoader(
    mismatched_inputs,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

print("Matched batches:", len(matched_loader))
print("Mismatched batches:", len(mismatched_loader))

Matched batches: 307
Mismatched batches: 308


## Evaluate INT8 on MNLI

In [22]:
def evaluate_mnli(model, dataloader):
    correct = 0
    total = 0

    model.eval()

    with torch.no_grad():
        for batch in dataloader:
            labels = batch.pop("labels")

            outputs = model(**batch)
            predictions = outputs.logits.argmax(dim=-1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return correct / total


matched_accuracy = evaluate_mnli(model_int8, matched_loader)
mismatched_accuracy = evaluate_mnli(model_int8, mismatched_loader)

print(f"Matched accuracy: {matched_accuracy:.4f}")
print(f"Mismatched accuracy: {mismatched_accuracy:.4f}")

Matched accuracy: 0.8438
Mismatched accuracy: 0.8483


## Save INT8 MNLI Results

In [23]:
mnli_results = {
    "model": "google-bert/bert-base-uncased",
    "seed": SEED,
    "precision": "dynamic_int8",
    "matched_accuracy": matched_accuracy,
    "mismatched_accuracy": mismatched_accuracy
}

mnli_results_path = f"../results/mnli/int8_seed{SEED}_results.json"

os.makedirs("../results/mnli", exist_ok=True)

with open(mnli_results_path, "w") as f:
    json.dump(mnli_results, f, indent=2)

print("Saved:", mnli_results_path)

Saved: ../results/mnli/int8_seed456_results.json
